# Lesson 1 - Exercise 1: Profile GPT-2 and Analyze Top Operators

**Goal:** Apply the profiling setup learned in the demo, run it on a smaller model (GPT-2), and practice accessing and interpreting the detailed profiler results to identify key performance bottlenecks.

**Task Overview:**
1.  Set the model name to `"gpt2"`.
2.  Re-run the CPU and GPU profiling sections, ensuring you capture results in `prof_cpu` and `prof_gpu`.
3.  Add code to print the detailed operator tables from the profiler results.
4.  Identify and list the Top 5 operators for both CPU and GPU.
5.  Interpret what these top operators likely represent.
6.  Compare the overall wall clock time of `gpt2` (this exercise) with `gpt2-medium` (from the demo or your own run if you did it).


## Imports

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.profiler
import time

## Load Model and Define Prompt

**TODO:**
- Set `model_name` variable to `"gpt2"`.
- The rest of the cell (loading tokenizer, model, setting pad token, and preparing inputs) can remain largely the same as the demo code.

In [2]:
# Define the model name for GPT-2 (the smallest variant, 124M parameters)
model_name = "gpt2"

print(f"Loading model and tokenizer for: {model_name}...")
try:
    # Load the tokenizer and model from the Hugging Face Hub (or local cache)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()  # inference mode (disables dropout)
except Exception as e:
    print(f"Error loading model {model_name}. Please ensure it's correct. Error: {e}")
    # If running in a restricted environment, you might need to use a pre-downloaded model or a different one.
    # For now, we'll stop if loading fails.
    raise

# Add a padding token if tokenizer doesn't have one
if tokenizer.pad_token is None:
    print("Setting pad_token to eos_token.")
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("Model loaded.")

# Prepare a sample prompt
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt")
num_new_tokens_to_generate = 50

print(f"Input prompt: '{prompt}'")
print(f"Generating {num_new_tokens_to_generate} new tokens.")


Loading model and tokenizer for: gpt2...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting pad_token to eos_token.
Model loaded.
Input prompt: 'The future of artificial intelligence is'
Generating 50 new tokens.


## CPU Profiling

**TODO:**
- Adapt the CPU profiling code from the demo.
- Ensure the inference is run within the `torch.profiler.profile` context.
- After the profiling block, add the code to print the top 5 CPU operators using `prof_cpu`.

In [3]:
print("\n--- Profiling on CPU ---")
cpu_device = torch.device("cpu")
model.to(cpu_device)
inputs_cpu = {k: v.to(cpu_device) for k, v in inputs.items()}

def run_cpu_inference(model_to_run, input_data, max_tokens, pad_id):
    """Greedy-decode `max_tokens` new tokens on CPU without tracking gradients."""
    with torch.no_grad():
        return model_to_run.generate(
            input_data["input_ids"],
            attention_mask=input_data.get("attention_mask"),
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=pad_id,
        )

print("Running inference on CPU and capturing profile...")
start_time_cpu_wall = time.time()

# Profile CPU activity only; shapes/memory recording is disabled to keep the overhead low.
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    record_shapes=False,
    profile_memory=False,
) as prof_cpu:
    # record_function labels the whole generation as one region in the trace
    with torch.profiler.record_function("model_inference_cpu"):
        run_cpu_inference(model, inputs_cpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

end_time_cpu_wall = time.time()
cpu_wall_time = end_time_cpu_wall - start_time_cpu_wall
print(f"CPU Wall clock time: {cpu_wall_time:.4f} seconds")

# Top 5 operators by self CPU time
print("\nCPU Profiler Analysis (Top 5 Operators by Self CPU Time):")
print(prof_cpu.key_averages().table(sort_by="self_cpu_time_total", row_limit=5))



--- Profiling on CPU ---
Running inference on CPU and capturing profile...


USDT:2026-08-16 15:14:45 69474:69474 SyncActivityProfilerHandler.cpp:52] profiler_start


USDT:2026-08-16 15:14:46 69474:69474 SyncActivityProfilerHandler.cpp:59] profiler_stop


CPU Wall clock time: 2.0510 seconds

CPU Profiler Analysis (Top 5 Operators by Self CPU Time):


-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                 Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          aten::addmm        41.31%     714.193ms        42.50%     734.741ms     306.142us          2400  
                                  model_inference_cpu        23.13%     399.863ms       100.00%        1.729s        1.729s             1  
                                             aten::mm        17.38%     300.561ms        17.39%     300.592ms       6.012ms            50  
                                            aten::cat         3.42%      59.150ms         3.65%      63.145ms      46.774us          1350  
                    

## GPU Profiling

**TODO:**
- Adapt the GPU profiling code from the demo.
- Include the check for CUDA availability (`torch.cuda.is_available()`).
- Remember the warm-up run and `torch.cuda.synchronize()` for accurate timing.
- Ensure the inference is run within the `torch.profiler.profile` context, capturing both CPU and CUDA activities.
- After the profiling block, add the code to print the top 5 GPU operators using `prof_gpu`.

In [4]:
gpu_wall_time = -1.0 # Initialize in case GPU is not available

if torch.cuda.is_available():
    print("\n--- Profiling on GPU ---")
    gpu_device = torch.device("cuda")
    model.to(gpu_device)
    inputs_gpu = {k: v.to(gpu_device) for k, v in inputs.items()}
    print(f"CUDA device found: {torch.cuda.get_device_name(gpu_device)}")

    def run_gpu_inference(model_to_run, input_data, max_tokens, pad_id):
        with torch.no_grad():
            # CUDA kernels are asynchronous: synchronize before/after so the timing
            # (and the profiler region) covers the whole generation.
            torch.cuda.synchronize()
            outputs = model_to_run.generate(
                input_data["input_ids"],
                attention_mask=input_data.get("attention_mask"),
                max_new_tokens=max_tokens,
                do_sample=False,
                pad_token_id=pad_id,
            )
            torch.cuda.synchronize()
            return outputs

    print("Performing GPU warm-up run...")
    # Warm-up: loads CUDA kernels / cuBLAS handles so they are not counted in the profile
    run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)
    print("Warm-up complete.")

    print("Running inference on GPU and capturing profile...")
    start_time_gpu_wall = time.time()

    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
        record_shapes=False,
        profile_memory=False,
    ) as prof_gpu:
        with torch.profiler.record_function("model_inference_gpu"):
            run_gpu_inference(model, inputs_gpu, num_new_tokens_to_generate, tokenizer.pad_token_id)

    end_time_gpu_wall = time.time()
    gpu_wall_time = end_time_gpu_wall - start_time_gpu_wall
    print(f"GPU Wall clock time: {gpu_wall_time:.4f} seconds")

    print("\nGPU Profiler Analysis (Top 5 Operators by Self CUDA Time):")
    print(prof_gpu.key_averages().table(sort_by="self_cuda_time_total", row_limit=5))

else:
    print("\nCUDA not available on this system. Skipping GPU profiling.")



CUDA not available on this system. Skipping GPU profiling.


## 4. Deliverables & Analysis

*Measured on an Intel Core i7-10610U (4 cores / 8 threads, AVX2, no GPU), PyTorch 2.13.0+cpu, transformers 5.15. Numbers will differ slightly between runs.*

**A. Top 5 CPU Operators:**
   1.  Operator Name: `aten::addmm` | Self CPU Time %: `41.3%` (714 ms, 2400 calls) | Likely Represents: the bias-added matrix multiplications of every `nn.Linear`/`Conv1D` layer — Q/K/V projection, attention output projection and the two MLP projections in each of the 12 blocks, called once per generated token.
   2.  Operator Name: `model_inference_cpu` | Self CPU Time %: `23.1%` (400 ms, 1 call) | Likely Represents: our own `record_function` region — its *self* time is Python / `generate()` orchestration overhead (sampling loop, logits processors, KV-cache bookkeeping) not attributed to any single tensor op.
   3.  Operator Name: `aten::mm` | Self CPU Time %: `17.4%` (301 ms, 50 calls) | Likely Represents: the LM-head projection (`hidden → vocab`, 768×50257) executed once per generated token (50 calls = 50 new tokens); it is a plain matmul without bias, hence `mm` rather than `addmm`.
   4.  Operator Name: `aten::cat` | Self CPU Time %: `3.4%` (59 ms, 1350 calls) | Likely Represents: concatenating the new key/value tensors onto the growing KV cache (2 per layer per step) and appending the sampled token to the sequence.
   5.  Operator Name: `aten::view` | Self CPU Time %: `1.7%` (29 ms, 10 253 calls) | Likely Represents: cheap reshapes used to split/merge attention heads and flatten activations — many calls, negligible individual cost.

**B. Top 5 GPU Operators (if CUDA was available):**
   *CUDA was not available on this machine, so the GPU cell was skipped. On a GPU run of the same code the top CUDA operators are typically:*
   1.  Operator Name: `ampere_sgemm_*` / `volta_sgemm_*` (cuBLAS GEMM kernels) | Self CUDA Time %: `~40–50%` | Likely Represents: the same linear-layer matmuls as `aten::addmm` on CPU.
   2.  Operator Name: `aten::addmm` / `aten::mm` (launcher ops) | Self CUDA Time %: `~10–20%` | Likely Represents: LM-head and projection matmuls.
   3.  Operator Name: `aten::softmax` / `aten::_softmax` | Self CUDA Time %: `~5%` | Likely Represents: attention probability normalisation.
   4.  Operator Name: elementwise kernels (`aten::add`, `aten::mul`, `aten::gelu`, `aten::native_layer_norm`) | Self CUDA Time %: `~5% each` | Likely Represents: residual adds, GELU activations and LayerNorm.
   5.  Operator Name: `aten::copy_` / `aten::cat` | Self CUDA Time %: `~3%` | Likely Represents: KV-cache concatenation and host↔device copies of the sampled token.

**C. Wall Clock Time Comparison (gpt2 vs gpt2-medium from demo):**
   - CPU Wall Time (gpt2 from this exercise): `≈ 2.05` seconds for 50 new tokens (124 M parameters)
   - CPU Wall Time (gpt2-medium, measured on the same machine with the same prompt/50 tokens): `≈ 4.5` seconds (355 M parameters)
   - GPU Wall Time (gpt2 from this exercise, if available): `N/A – no CUDA device`
   - GPU Wall Time (gpt2-medium from demo, approx., if available): `N/A here (demo reference ≈ 0.5–1.5 s)`
   
   - Did `gpt2` (smaller model) run faster than `gpt2-medium` (larger model) on both CPU and GPU as expected? `Yes on CPU: ~2.2x faster with ~2.9x fewer parameters. The speed-up is smaller than the parameter ratio because per-token overhead in generate() (Python loop, cache handling, tiny kernels) is roughly constant and does not shrink with model size — it accounts for ~25% of the small model's time. On a GPU the gap would shrink further, since decoding at batch size 1 is launch/latency-bound rather than compute-bound.`

**D. Brief Interpretation of Top Operators:**
   - What kind of operations generally dominate the top spots on both CPU and GPU for this Transformer model? `Dense matrix multiplications (aten::addmm / aten::mm → cuBLAS GEMMs on GPU) coming from the linear projections in attention and the MLP, plus the LM head, account for ~60% of self time. Attention-specific ops (softmax, bmm) and elementwise ops (LayerNorm, GELU, residual adds) come next; memory-movement ops (cat/view/copy) are frequent but individually cheap.`
   - Were there any surprising operators in the top 5 for either CPU or GPU? `Yes: (1) the record_function region itself shows ~23% self time — a reminder that at batch size 1 the generate() loop's Python overhead is a first-class cost, which motivates CUDA graphs / compiled decoding loops. (2) aten::mm (LM head) alone is ~17%: with a 50k vocabulary the output projection is as expensive as several transformer layers, so its cost per token is comparable to the whole rest of the model.`
